In [ ]:
!pip install -r requirements.txt

In [ ]:
import json
import faiss
import numpy as np
from huggingface_hub import hf_hub_download
from transformers import AutoModel, AutoTokenizer
import torch
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

In [ ]:
# Hugging Face 모델 설정
MODEL_NAME = "jinaai/jina-embeddings-v3"
# hf_hub_download(repo_id=MODEL_NAME, filename="config.json")  # 모델 파일 강제 다운로드

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, trust_remote_code=True).to("cuda")

In [ ]:
def load_json_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

In [ ]:
# Word2Vec 모델 학습 함수
def train_word2vec(texts, vector_size=100, window=5, min_count=1):
    tokenized_texts = [simple_preprocess(text) for text in texts]
    model = Word2Vec(sentences=tokenized_texts, vector_size=vector_size, window=window, min_count=min_count, workers=4)
    return model

In [ ]:
# Doc2Vec 모델 학습 함수
def train_doc2vec(texts, vector_size=100, window=5, min_count=1):
    tagged_data = [TaggedDocument(words=simple_preprocess(text), tags=[str(i)]) for i, text in enumerate(texts)]
    model = Doc2Vec(vector_size=vector_size, window=window, min_count=min_count, workers=4, epochs=20)
    model.build_vocab(tagged_data)
    model.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)
    return model

In [ ]:
def compute_sentence_embedding_w2v(model, text):
    words = simple_preprocess(text)
    word_vectors = [model.wv[word] for word in words if word in model.wv]
    return np.mean(word_vectors, axis=0) if word_vectors else np.zeros(model.vector_size)

In [ ]:
# 문장의 임베딩 계산 함수 (Doc2Vec)
def compute_sentence_embedding_d2v(model, text):
    words = simple_preprocess(text)
    return model.infer_vector(words)

In [ ]:
def compute_embeddings(texts, batch_size=1):
    # texts 3800개가 들어옴
    embeddings = []
    for i in range(0, len(texts), batch_size):
        # 각 배치 당 8개씩의 text가 들어옴
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to("cuda")  # 입력을 GPU로 이동
        with torch.no_grad():
            outputs = model(**inputs)
        batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()  # CLS 토큰 임베딩을 CPU로 이동 후 변환
        embeddings.append(batch_embeddings)
    return np.vstack(embeddings)

In [ ]:
def compute_cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

In [ ]:
# JSON 데이터 로드 (예시 파일 경로)
data = load_json_data("gpt_results_2503.json")

# Original 및 Summary 텍스트 추출
original_texts = [entry["original_text"] for entry in data]
summary_texts = [entry["summary_text"] for entry in data]
qa_texts = [entry["qa_text"] for entry in data]

print(len(original_texts))
print(len(summary_texts))
print(len(qa_texts))

# Word2Vec

In [ ]:
# Word2Vec 모델 학습
all_texts = original_texts + summary_texts
w2v_model = train_word2vec(all_texts)

# Word2Vec 임베딩 생성
original_embeddings_w2v = np.array([compute_sentence_embedding_w2v(w2v_model, text) for text in original_texts])
summary_embeddings_w2v = np.array([compute_sentence_embedding_w2v(w2v_model, text) for text in summary_texts])

In [ ]:
# 코사인 유사도 계산 (Word2Vec)
similarities_w2v = [
    compute_cosine_similarity(original_embeddings_w2v[i], summary_embeddings_w2v[i])
    for i in range(len(original_embeddings_w2v))
]

# Doc2Vec

In [ ]:
from gensim.models import Word2Vec, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from gensim.utils import simple_preprocess

In [ ]:
# Doc2Vec 모델 학습
all_texts = original_texts + summary_texts
d2v_model = train_doc2vec(all_texts)

In [ ]:
# Doc2Vec 임베딩 생성
original_embeddings_d2v = np.array([compute_sentence_embedding_d2v(d2v_model, text) for text in original_texts])
summary_embeddings_d2v = np.array([compute_sentence_embedding_d2v(d2v_model, text) for text in summary_texts])

In [ ]:
# 코사인 유사도 계산 (Doc2Vec)
similarities_d2v = [
    compute_cosine_similarity(original_embeddings_d2v[i], summary_embeddings_d2v[i])
    for i in range(len(original_embeddings_d2v))
]

In [ ]:

output_data = [{
  "original_id": data[i]["original_id"],
  "original_text": data[i]["original_text"],
  "summary_text": data[i]["summary_text"],
  "cosine_similarity_w2v": float(similarities_w2v[i]),
  "cosine_similarity_d2v": float(similarities_d2v[i])
} for i in range(len(data))]

In [ ]:
# 결과 JSON 파일에 저장
with open("similarity_results.json", "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=4, ensure_ascii=False)

# Huggingface Embeddings model

In [ ]:
# 임베딩 생성
original_embeddings = compute_embeddings(original_texts)
summary_embeddings = compute_embeddings(summary_texts)

In [ ]:
# 코사인 유사도 계산
similarities = [
    compute_cosine_similarity(original_embeddings[i], summary_embeddings[i])
    for i in range(len(original_embeddings))
]

In [ ]:
# 결과 JSON 파일에 저장 (only jina)
output_data = [{
  "id": data[i]["id"],
  "original_text": data[i]["original_text"],
  "summary_text": data[i]["summary_text"],
  "cosine_similarity_jina": str(similarities[i]),
} for i in range(len(data))]

with open("similarity_results_0328.json", "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=4, ensure_ascii=False)

## Cosine Similarity 필터링

In [ ]:
low_data = []
for data in output_data:
    if float(data["cosine_similarity_jina"]) <= 0.8:
        low_data.append(data)

with open("low_cosine_similarity.json", "w", encoding="utf-8") as f:
    json.dump(low_data, f, indent=4, ensure_ascii=False)

print(len(low_data))
        

# 시각화

In [ ]:
!pip install matplotlib
!pip install seaborn

# Embedding Models

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 히스토그램 및 KDE 플롯
plt.figure(figsize=(6,4))
plt.tight_layout()
sns.histplot(similarities, bins=30, kde=True, color="blue", edgecolor="black")

plt.xlabel("Cosine Similarity", fontsize=14)
plt.ylabel("Frequency", fontsize=14)
plt.title("Jina - Distribution of Cosine Similarities", fontsize=16)
plt.grid(True)
plt.savefig('figure3.pdf')
plt.show()

## Word2Vec

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 히스토그램 및 KDE 플롯
plt.figure(figsize=(10, 6))
sns.histplot(similarities_w2v, bins=30, kde=True, color="blue", edgecolor="black")

plt.xlabel("Cosine Similarity", fontsize=14)
plt.ylabel("Frequency", fontsize=14)
plt.title("Word2Vec - Distribution of Cosine Similarities", fontsize=16)
plt.grid(True)
plt.show()

## Doc2Vec

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 히스토그램 및 KDE 플롯
plt.figure(figsize=(10, 6))
sns.histplot(similarities_d2v, bins=30, kde=True, color="blue", edgecolor="black")

plt.xlabel("Cosine Similarity", fontsize=14)
plt.ylabel("Frequency", fontsize=14)
plt.title("Doc2Vec - Distribution of Cosine Similarities", fontsize=16)
plt.grid(True)
plt.show()

## Cumulative Distribution Function

In [ ]:
import json
import matplotlib.pyplot as plt
from cdf_plot_style import plot_model_comparison_cdf

def load_doc_cosine(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    return [v["_overall_"]["cosine_sim"] for v in raw.values()]

# 전체 코퍼스 재생성본 (run_cosine_stats.sh, jina-v3, pooling=cls) 사용.
# 이 노트북 자체의 임베딩 파이프라인은 cosine <= 0.8 tail만 저장했던 구버전이라 쓰지 않는다.
data = {
    "gpt-4o-mini": load_doc_cosine("notes/doc_cosine_gpt4omini.json"),
    "gpt-3.5-turbo": load_doc_cosine("notes/doc_cosine_gpt35.json"),
}
plot_model_comparison_cdf(data, xlabel="Score", savepath="figures/figure3.pdf")
plt.show()